# 01 — Korpus & Annotation

Phase 1 (Korpus + Bias-Notiz) und Phase 2 (κ + drei Edge Cases) leben in diesem Notebook. Pipeline → `02_extract.ipynb`, Eval → `03_eval.ipynb`, Frontier → `04_frontier_compare.ipynb`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum (Phase 1) | 2026-05-04 |
| Datum (Phase 2) | _YYYY-MM-DD_ |
| Korpus-Datei | `daten/eigener_korpus.jsonl` |
| Anzahl Anzeigen im Korpus | 90 |
| Genutzte Suchanfragen (Phase 1) | `Fachinformatiker Daten- und Prozessanalyse` (25), `Data Scientist` (20), `Datenanalyst` (20→15 verfügbar), `Business Intelligence` (15), `Data Engineer` (15) |
| Pair-Partner:in (Phase 2) | _ |
| 12 gemeinsame Anzeigen-IDs | _ |

## Phase 1 — Korpus aufbauen + inspizieren

### Block 1.1 — Korpus von der Bundesagentur-API ziehen

Die [Jobsuche-API der Bundesagentur](https://github.com/bundesAPI/jobsuche-api) liefert Suchergebnisse + Detail-Beschreibungen. Header `X-API-Key: jobboerse-jobsuche` ist öffentlich dokumentiert.

Zwei Endpoints:
- `GET /pc/v4/jobs?was=…&page=…&size=…` — paginierte Suche, liefert Liste von Stellenangeboten mit `refnr`
- `GET /pc/v4/jobdetails/{hashId}` — Detail-Beschreibung pro Anzeige. `hashId` = base64(refnr) ohne Padding

Pro Anzeige speichern wir mind. `refnr`, `titel`, `firma`, `text` plus die Strukturfelder, die die API ohnehin mitschickt (für Block 1.2).

In [ ]:
import base64
import json
import time
from pathlib import Path

import requests

API_BASE = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service"
HEADERS = {"X-API-Key": "jobboerse-jobsuche"}

KORPUS_PATH = Path("../daten/eigener_korpus.jsonl")
KORPUS_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def search(was: str, wo: str | None = None, size: int = 50, page: int = 1) -> dict:
    params = {"was": was, "page": page, "size": size}
    if wo:
        params["wo"] = wo
    r = requests.get(f"{API_BASE}/pc/v4/jobs", params=params, headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()


def detail(refnr: str) -> dict:
    # API erwartet base64-encoded refnr ohne Padding
    hash_id = base64.b64encode(refnr.encode("utf-8")).decode("ascii").rstrip("=")
    r = requests.get(f"{API_BASE}/pc/v4/jobdetails/{hash_id}", headers=HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()


def extrahiere_treffer(such_response: dict, suchbegriff: str) -> list[dict]:
    """Holt aus einer Such-Response die Mindestfelder + behält Strukturfelder fürs Bias-Audit."""
    out = []
    for st in such_response.get("stellenangebote", []):
        out.append({
            "refnr": st.get("refnr"),
            "titel": st.get("titel") or st.get("beruf"),
            "firma": st.get("arbeitgeber"),
            "ort": (st.get("arbeitsort") or {}).get("ort"),
            "plz": (st.get("arbeitsort") or {}).get("plz"),
            "region": (st.get("arbeitsort") or {}).get("region"),
            "eintrittsdatum": st.get("eintrittsdatum"),
            "aktuelleVeroeffentlichungsdatum": st.get("aktuelleVeroeffentlichungsdatum"),
            "externeUrl": st.get("externeUrl"),
            "hashId": st.get("hashId"),
            "_suchbegriff": suchbegriff,
        })
    return out


def reichere_mit_text_an(eintrag: dict, sleep: float = 0.4) -> dict:
    """Holt die Detail-Beschreibung und hängt sie als `text` an den Eintrag.

    Hinweis: der Detail-Endpoint nennt das Beschreibungs-Feld `stellenangebotsBeschreibung`
    (nicht `stellenbeschreibung`) — und liefert weitere Strukturfelder wie
    `verguetungsangabe`, `vertragsdauer`, `arbeitszeitVollzeit`, die für das
    Bias-Audit (Block 1.2) interessant sind.
    """
    try:
        d = detail(eintrag["refnr"])
    except requests.HTTPError as e:
        eintrag["text"] = ""
        eintrag["_detail_fehler"] = str(e)
        return eintrag
    eintrag["text"] = d.get("stellenangebotsBeschreibung") or ""
    eintrag["stellenangebotsTitel"] = d.get("stellenangebotsTitel")
    eintrag["verguetungsangabe"] = d.get("verguetungsangabe")
    eintrag["vertragsdauer"] = d.get("vertragsdauer")
    eintrag["arbeitszeitVollzeit"] = d.get("arbeitszeitVollzeit")
    eintrag["quereinstiegGeeignet"] = d.get("quereinstiegGeeignet")
    eintrag["stellenangebotsart"] = d.get("stellenangebotsart")
    time.sleep(sleep)
    return eintrag

In [ ]:
# Mehrere Suchanfragen für einen Mix — ändere/erweitere die Liste, falls dein Korpus zu einseitig wird.
# Ziel: ≥ 30 Anzeigen nach Dedup. Ausbildung + Festanstellung + verschiedene Berufsbezeichnungen mischen.
SUCHANFRAGEN = [
    {"was": "Fachinformatiker Daten- und Prozessanalyse", "size": 25},
    {"was": "Data Scientist", "size": 20},
    {"was": "Datenanalyst", "size": 20},
    {"was": "Business Intelligence", "size": 15},
    {"was": "Data Engineer", "size": 15},
]

treffer_roh: list[dict] = []
for q in SUCHANFRAGEN:
    resp = search(was=q["was"], size=q["size"])
    n_max = resp.get("maxErgebnisse", 0)
    treffer = extrahiere_treffer(resp, q["was"])
    print(f"  '{q['was']}': {len(treffer)} Treffer (von {n_max} verfügbar)")
    treffer_roh.extend(treffer)
    time.sleep(0.5)

print(f"\nGesamt vor Dedup: {len(treffer_roh)}")

# Dedup über refnr — eine Anzeige kann in mehreren Suchen auftauchen
gesehen: set[str] = set()
treffer_dedup: list[dict] = []
for t in treffer_roh:
    if not t["refnr"] or t["refnr"] in gesehen:
        continue
    gesehen.add(t["refnr"])
    treffer_dedup.append(t)

print(f"Nach Dedup: {len(treffer_dedup)}")
assert len(treffer_dedup) >= 30, f"Zu wenig Anzeigen ({len(treffer_dedup)}) — Suchanfragen erweitern."

In [ ]:
# Detail-Texte holen (mit Pause zwischen Requests, sonst rate-limited die API)
korpus: list[dict] = []
for i, t in enumerate(treffer_dedup, 1):
    angereichert = reichere_mit_text_an(t)
    korpus.append(angereichert)
    if i % 10 == 0:
        print(f"  {i}/{len(treffer_dedup)} Detail-Anzeigen geladen")

n_leer = sum(1 for k in korpus if not k.get("text"))
print(f"\nFertig: {len(korpus)} Anzeigen, davon {n_leer} ohne Beschreibungstext.")

# Anzeigen ohne Text fliegen raus — die taugen für Annotation nichts
korpus = [k for k in korpus if k.get("text")]
print(f"Nach Text-Filter: {len(korpus)} Anzeigen.")

In [ ]:
# JSONL schreiben (siehe CHEATSHEETS/jsonl.md)
with KORPUS_PATH.open("w", encoding="utf-8") as f:
    for eintrag in korpus:
        f.write(json.dumps(eintrag, ensure_ascii=False) + "\n")

print(f"{len(korpus)} Anzeigen geschrieben nach {KORPUS_PATH}")

### Block 1.2 — Korpus inspizieren

Vier Blicke aufs Material: Verteilungen, mehrfach genannte Firmen, Berufsbezeichnungen, Strukturfeld-Coverage. Output dieser Zellen ist die Faktenbasis für die Bias-Notiz unten.

In [ ]:
import pandas as pd

df = pd.read_json(KORPUS_PATH, lines=True)
print(f"Korpus-Größe: {len(df)} Anzeigen")
df.head(2)

In [ ]:
# Längen-Verteilung der Beschreibungstexte (Zeichen)
df["text_len"] = df["text"].str.len()
print(df["text_len"].describe().round(0))
df["text_len"].plot.hist(bins=20, title="Verteilung Textlängen (Zeichen)");

In [ ]:
# Firmen-Häufigkeit — wer taucht mehrfach auf?
firmen = df["firma"].value_counts()
print(f"Anzahl unterschiedlicher Firmen: {firmen.size}")
print("\nTop 10 Firmen:")
print(firmen.head(10))
print(f"\nFirmen mit ≥2 Anzeigen: {(firmen >= 2).sum()}")

In [ ]:
# Wie verteilen sich die Anzeigen über die Suchanfragen?
print("Anzeigen pro Suchanfrage (vor Dedup gezählt nach _suchbegriff):")
print(df["_suchbegriff"].value_counts())

In [ ]:
# Berufsbezeichnungen — was ist im Titel-Feld unterschiedlich?
print(f"Unterschiedliche Titel: {df['titel'].nunique()}")
print("\nTop 10 Titel:")
print(df["titel"].value_counts().head(10))

In [ ]:
# Regionale Verteilung — bundesweit oder klumpig?
print("Top 10 Orte:")
print(df["ort"].value_counts().head(10))
print("\nTop 10 Regionen:")
print(df["region"].value_counts().head(10))

In [ ]:
# Strukturfeld-Coverage: welche API-Felder sind wie oft befüllt?
strukturfelder = [
    "firma", "ort", "plz", "region", "eintrittsdatum",
    "aktuelleVeroeffentlichungsdatum", "externeUrl",
    "arbeitszeitVollzeit", "vertragsdauer", "verguetungsangabe",
    "quereinstiegGeeignet", "stellenangebotsart",
]
vorhanden = [c for c in strukturfelder if c in df.columns]
coverage = (df[vorhanden].notna() & (df[vorhanden].astype(str) != "") & (df[vorhanden].astype(str) != "null")).mean().round(2) * 100
print("Coverage der Strukturfelder (% befüllt):")
print(coverage.sort_values(ascending=False))

print()
print("--- Verguetungsangabe-Verteilung ---")
print(df["verguetungsangabe"].value_counts(dropna=False))
print()
print("--- Vertragsdauer-Verteilung ---")
print(df["vertragsdauer"].value_counts(dropna=False))
print()
print("--- Stellenangebotsart-Verteilung ---")
print(df["stellenangebotsart"].value_counts(dropna=False))


### Block 1.2b — Freitext (`text`) inspizieren

Die Strukturfelder sind dünn (siehe Cell oben — `verguetungsangabe` 82 % `KEINE_ANGABEN`, `vertragsdauer` 54 % `KEINE_ANGABE`). Alles Schema-Relevante steckt im Markdown-Freitext (`stellenangebotsBeschreibung`). Diese Zelle scannt ihn nach Schlüsselwörtern für die sechs Schema-Felder + häufigsten Section-Markern und gibt damit eine zweite Faktenbasis für die Bias-Notiz.

In [ ]:
import re
from collections import Counter

def hat(pat: str) -> int:
    """Anzahl Anzeigen, in deren `text` das Regex matcht."""
    rx = re.compile(pat, re.IGNORECASE)
    return sum(1 for t in df["text"] if rx.search(t))

n = len(df)
print("Korpus n =", n)
print()

# 1. Schema-Felder — Schlüsselwörter pro Schema-Feld
print("--- homeoffice ---")
for label, pat in [
    ("homeoffice (Wort)", r"home\s*-?\s*office|homeoffice"),
    ("remote",            r"\bremote\b"),
    ("hybrid / mobil",    r"\bhybrid\b|mobiles\s+arbeiten"),
    ("Präsenz / vor Ort", r"pr(ä|ae)senzpflicht|vor\s+ort"),
]:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")

print()
print("--- vertragsart ---")
for label, pat in [
    ("Ausbildung / Azubi",  r"ausbildung|azubi"),
    ("Werkstudent",         r"werkstud"),
    ("Praktikum",           r"praktik"),
    ("Trainee",             r"\btrainee\b"),
    ("unbefristet",         r"\bunbefristet"),
    ("befristet",           r"\bbefristet\b(?!.{0,5}unbefristet)"),
]:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")

print()
print("--- erfahrungslevel ---")
for label, pat in [
    ("Senior",                  r"\bsenior\b"),
    ("Junior",                  r"\bjunior\b"),
    ('"X Jahre (Berufs)Erfahrung"', r"\d+\+?\s*jahr(e)?\s+(berufs)?erfahrung"),
    ("Berufseinsteiger",        r"berufseinsteiger|einsteiger"),
]:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")

print()
print("--- gehalt (Freitext, da Strukturfeld bei 82% KEINE_ANGABEN) ---")
for label, pat in [
    ("Schlüsselwort Gehalt/Vergütung", r"gehalt|verg(ü|ue)tung|bezahlung"),
    ("konkrete Zahl mit € / EUR",      r"\d{1,3}[\.,]?\d{3,4}\s*(€|eur|euro)"),
    ('"ab X €"',                       r"ab\s+\d{1,3}[\.,]?\d{3}\s*(€|eur)"),
    ("Tarif (TVöD / TV-L / nach Tarif)", r"tarif|tv-?l\b|tv(ö|oe)d"),
]:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")

print()
print("--- skills_top3 (Tools/Sprachen) ---")
skills = [
    ("Python", r"\bpython\b"), ("SQL", r"\bsql\b"), ("Java", r"\bjava\b(?!script)"),
    ("JavaScript / TypeScript", r"javascript|\btypescript\b"), ("Excel", r"\bexcel\b"),
    ("Power BI", r"power\s*bi"), ("Tableau", r"tableau"),
    ("Azure", r"\bazure\b"), ("AWS", r"\baws\b|amazon\s+web"), ("GCP", r"\bgcp\b|google\s+cloud"),
    ("Snowflake", r"snowflake"), ("Databricks", r"databricks"),
    ("Docker", r"\bdocker\b"), ("Kubernetes / K8s", r"kubernetes|\bk8s\b"),
    ("Pandas", r"\bpandas\b"), ("ETL", r"\betl\b"),
    ("Machine Learning", r"machine\s+learning|\bml\b"),
    ("dbt", r"\bdbt\b"), ("Spark", r"\bspark\b"), ("Git", r"\bgit\b(?!hub|lab)"),
]
for label, pat in skills:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")

# 2. Konkrete Gehaltsangaben aus dem Text — wer hat sie überhaupt?
print()
print("--- Anzeigen mit konkreter EUR-Zahl im Text (max 15) ---")
gehalt_re = re.compile(r"(\d{1,3}[\.,]?\d{3,4})\s*(?:€|eur|euro)", re.IGNORECASE)
mit_zahl = []
for _, row in df.iterrows():
    treffer = gehalt_re.findall(row["text"])
    if treffer:
        mit_zahl.append((row["refnr"], row["stellenangebotsart"], treffer[:3]))
print(f"  {len(mit_zahl)}/{n} Anzeigen haben mind. eine €-Zahl im Text")
for refnr, art, tr in mit_zahl[:15]:
    print(f"  {refnr}  [{art}]  → {tr}")

# 3. Häufigste Markdown-Headings (Section-Marker im Text)
print()
print("--- Top 15 Markdown-Headings (## bis ######) ---")
heading_re = re.compile(r"(?m)^#{1,6}\s+(.+?)\s*$")
hc: Counter = Counter()
for t in df["text"]:
    for h in heading_re.findall(t):
        hc[h.strip().lower()[:60]] += 1
for h, c in hc.most_common(15):
    print(f"  {c:3d}  {h}")

# 4. Dauer-Hinweise (relevant für Ausbildung/Befristung)
print()
print("--- Dauer / Eintritt ---")
for label, pat in [
    ('"X Jahre" (Ausbildungs-/Vertragsdauer)', r"\b(2|2,5|3|3,5)\s*jahr"),
    ("Eintritt 2025/2026",                     r"\b202[5-9]\b"),
    ("Vollzeit",                               r"\bvollzeit\b"),
    ("Teilzeit",                               r"\bteilzeit\b"),
]:
    h = hat(pat); print(f"  {h:3d}/{n}  {h/n:5.0%}  {label}")


### Bias-Notiz (5–7 Sätze, eigene Worte)

Mein Korpus hat **90 Anzeigen** — gut über fünf Suchanfragen verteilt, aber alle aus dem **Daten/IT-Umfeld** (Fachinformatiker DPA, Data Scientist, Datenanalyst, BI, Data Engineer); jede Aussage meiner Pipeline gilt damit ausschließlich für diese Berufsfamilie und ist nicht auf Pflege, Vertrieb oder Handwerk übertragbar. Die Suche `Fachinformatiker Daten- und Prozessanalyse` zieht überproportional viele **Ausbildungen** rein — `stellenangebotsart=AUSBILDUNG` bei 25/90 (28 %) — was das Schema-Feld `vertragsart` einseitig Richtung `ausbildung`/`festanstellung` schiebt und Werkstudenten/Praktika fast komplett fehlen lässt (nur 1 PRAKTIKUM_TRAINEE, kein WERKSTUDENT, 82/90 = 91 % Vollzeit). Regional klumpen NRW (19), Bayern (14) und BW (12) auf zusammen **50 %** des Korpus, gleichzeitig sind ~16 Anzeigen aus Österreich (Wien 9, Oberösterreich 4, Salzburg 3) drin — was κ-Vergleiche mit Partner:innen verzerrt, falls deren Korpus ausschließlich DE-Anzeigen enthält. Beim **Gehalt** liefert das Strukturfeld kaum Signal: `verguetungsangabe=KEINE_ANGABEN` bei 74/90 (82 %), nur 13/90 mit `JAHRESGEHALT` und 3 mit Ausbildungstarif — Gehalt müssen wir also fast immer aus dem Freitext extrahieren, und mein Hand-Gold wird viele `null`-Werte enthalten (was das Feld später in der Eval künstlich „leicht" wirken lässt). **Plattform-Effekt Bundesagentur:** Personaldienstleister wie FERCHAU (3 Anzeigen) und Hays sowie öffentlich-rechtliche/Versicherungs-Arbegeber (HUK-COBURG, Wien Energie, GVV) sind sichtbar überrepräsentiert — schnell wachsende Tech-Unternehmen, die Stellen primär auf LinkedIn ausschreiben, fehlen praktisch komplett. Schließlich: 85 unterschiedliche Firmen bei 90 Anzeigen heißt fast keine Firmen-Duplikate — gut für Vielfalt, aber bedeutet auch, dass jede einzelne Firma nur eine sehr enge Anzeigen-Stilprobe liefert.

## Phase 2 — κ-Tabelle + drei Edge Cases